# Phần preprocessing

Thư viện

In [13]:
import os
import numpy as np
import pandas as pd

Định nghĩa các thông số

In [14]:
step = 20 # Time_step
X = [] # Data sau cùng
Y = [] # Label (1 là tục, 0 là không tục)

Lấy data tục

In [15]:
# Lấy ra các file trong thư mục
list_files_csv = os.listdir('DATA_TUC')

for file_csv in list_files_csv:
    # Mở từng file csv ra
    with open('DATA_TUC/' + file_csv, 'r') as f:
        # Chuyển data về dạng numpy để tính toán
        data = pd.read_csv(f).to_numpy()
        # Lấy data, label
        # Lấy một lượng step dòng và bước nhảy là 1
        for i in range(step, data.shape[0]):
            X.append(data[i - step: i, :])
            Y.append(1)

In [16]:
len(Y)

954

Lấy data không tục

In [17]:
list_files_csv = os.listdir('DATA_KHONG_TUC')
for file_csv in list_files_csv:
    with open('DATA_KHONG_TUC/' + file_csv, 'r') as f:
        data = pd.read_csv(f).to_numpy()
        for i in range(step, data.shape[0]):
            X.append(data[i - step: i, :])
            Y.append(0)

Chuyển về numpy

In [18]:
X = np.array(X)
Y = np.array(Y)
print(X.shape, Y.shape)

(2202, 20, 80) (2202,)


# Phần này là training bằng tensorflow, muốn training bằng pytorch thì chuyển numpy sang tensor và training    

Chia dữ liệu

In [19]:
import pickle
pickle.dump(X, open('X.pkl', 'wb'))
pickle.dump(Y, open('Y.pkl', 'wb'))

In [20]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.2)


Tạo model

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, Dropout, BatchNormalization, Bidirectional, GlobalAveragePooling1D, MultiHeadAttention, LayerNormalization, Add, Input
)
from tensorflow.keras.models import Model

# Tham số mô hình
TIME_STEPS = step  # Chiều dài chuỗi đầu vào
FEATURES = 80    # Số đặc trưng mỗi bước thời gian

# Input Layer
inputs = Input(shape=(TIME_STEPS, FEATURES))

# LSTM Layer 1
x = Bidirectional(LSTM(128, activation='tanh', return_sequences=True))(inputs)

# LSTM Layer 2
x = Bidirectional(LSTM(256, activation='tanh', return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

# LSTM Layer 4
x = Bidirectional(LSTM(512, activation='tanh', return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

# GlobalAveragePooling1D để lấy thông tin quan trọng nhất
x = GlobalAveragePooling1D()(x)
x = Dropout(0.2)(x)

# Dense Layers
x = Dense(16, activation="tanh")(x)
x = Dropout(0.2)(x)

# Output Layer (Binary Classification)
outputs = Dense(1, activation="sigmoid")(x)

# Tạo model
model = Model(inputs, outputs)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# In tóm tắt mô hình
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20, 80)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 20, 256)        │       214,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 20, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 20, 512)        │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 20, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 20, 1024)       │     4,198,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 20, 1024)       │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 20, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │        16,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,485,601 (20.93 MB)

 Trainable params: 5,482,529 (20.91 MB)

 Non-trainable params: 3,072 (12.00 KB)

Training

In [22]:
history = model.fit(xtrain, ytrain, batch_size= 16, epochs= 20, validation_data= (xtest, ytest))

Epoch 1/20


KeyboardInterrupt: 

Lưu model

In [ ]:
model.save('model_lstm.h5')